# Task 11: DiffusionGemma website builder and speed test

Run the cells **one at a time, top to bottom**. You need a GPU with **80 GB** of memory
(Runtime > Change runtime type > **A100** with High-RAM, or H100). The model is about 50 GB in bf16.

The safety timer in step 1b shuts everything down after 4 hours, and the last cell shuts it down as soon as you are done.
You can also stop it any time with **Runtime > Disconnect and delete runtime**.

## 1. Check the GPU and the disk

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /content | tail -1

## 1b. Safety timer (run this right after step 1)
Shuts the runtime down by itself after `HOURS_LIMIT` hours, even if you walk away or forget, so it cannot keep using compute units.
An A100 uses roughly 5 to 8 units per hour, so 4 hours is about 20 to 32 units. The whole notebook should take about 2 hours.

In [ ]:
HOURS_LIMIT = 4

import threading, time, datetime
from google.colab import runtime

def _shut_down():
    print("Time limit reached: shutting the runtime down to save compute units.")
    runtime.unassign()

_timer = threading.Timer(HOURS_LIMIT * 3600, _shut_down)
_timer.daemon = True
_timer.start()
print("Safety timer started. The runtime will shut itself down at",
      (datetime.datetime.now() + datetime.timedelta(hours=HOURS_LIMIT)).strftime("%H:%M"), "(Colab's clock, UTC).")

## 2. Install the libraries (a few minutes)

In [ ]:
!pip install -q -U transformers accelerate gradio
import subprocess, sys
ok = subprocess.run([sys.executable, "-c", "from transformers import DiffusionGemmaForBlockDiffusion"]).returncode == 0
if not ok:
    print("The released transformers does not have DiffusionGemma yet, installing the newest version from GitHub...")
    !pip install -q git+https://github.com/huggingface/transformers
    ok = subprocess.run([sys.executable, "-c", "from transformers import DiffusionGemmaForBlockDiffusion"]).returncode == 0
print("DiffusionGemma available:", ok)

## 3. Log in to Hugging Face
Paste a token from https://huggingface.co/settings/tokens (a *Read* token is enough).

In [ ]:
from huggingface_hub import login
login()

## 4. Upload `11-diffusion-demo.zip`
A file picker appears under the cell. Pick the zip from your Downloads folder.

In [ ]:
from google.colab import files
files.upload()
!unzip -o -q 11-diffusion-demo.zip
%cd /content/11-diffusion-demo
!ls

## 5. Load DiffusionGemma (downloads about 50 GB the first time, so this takes a while)

In [ ]:
import time, core
t0 = time.time()
model, processor = core.load_model("diffusion")
print(f"Loaded in {time.time() - t0:.0f} s")
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

## 6. Quick test: one short answer

In [ ]:
for frame in core.stream_generate(model, processor, "In two sentences, what is a diffusion language model?",
                                  max_new_tokens=256, as_page=False):
    if frame["kind"] == "draft" and frame["step"] % 8 == 1:
        print(f"step {frame['step']:>2}:", repr(frame["text"][:80]))
    elif frame["kind"] in ("commit", "error"):
        print("\nFINAL:", frame.get("text") or frame.get("message"))
    elif frame["kind"] == "done":
        print({k: v for k, v in frame.items() if k != "kind"})

## 7. The demo app
Click the **public URL** it prints (ends in `gradio.live`). Try a few requests and take screenshots for the README.
The app keeps running in the background, so you can go on to step 8 while it is open.

In [ ]:
import app
app.launch(model, processor, share=True)

## 8. Speed test, DiffusionGemma (6 prompts x 3 runs)

In [ ]:
import benchmark
benchmark.run("diffusion", model, processor, max_new_tokens=1024, repeats=3)

## 9. Swap to the regular Gemma 4 26B-A4B
This closes the app, frees the GPU, and deletes the downloaded models so the disk has room for the second one (your Hugging Face login is kept).

In [ ]:
import shutil, os
app.stop()
del model, processor
core.free_gpu()
for d in ("hub", "xet"):
    shutil.rmtree(os.path.expanduser("~/.cache/huggingface/" + d), ignore_errors=True)
print(f"Free disk: {shutil.disk_usage('/content').free / 1e9:.0f} GB")
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

### 9b. If the GPU still shows more than 5000 MiB used above
Something still holds the old model. Do **Runtime > Restart session**, then run step **1b** (the safety timer) again, then this cell, then go to step 10.
If it showed about 0 MiB, skip this cell.

In [ ]:
%cd /content/11-diffusion-demo
import time, core, benchmark, torch
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv
!ls results

## 10. Load the regular model and run the same speed test

In [ ]:
t0 = time.time()
model, processor = core.load_model("regular")
print(f"Loaded in {time.time() - t0:.0f} s")
benchmark.run("regular", model, processor, max_new_tokens=1024, repeats=3)

## 11. Compare and download the results

In [ ]:
from google.colab import files
benchmark.combine()
!cat results/bench_summary.csv
!nvidia-smi --query-gpu=name --format=csv,noheader > results/gpu.txt
!pip list 2>/dev/null | grep -iE "^(torch|transformers|accelerate|gradio) " > results/versions.txt
!zip -q -r /content/11-results.zip results
files.download("/content/11-results.zip")

## 12. Shut down
Wait until `11-results.zip` has finished downloading in your browser, then run this cell. It deletes the runtime so it stops using compute units.

In [ ]:
from google.colab import runtime
runtime.unassign()